# NB31 · Real-World Programs

`advanced/` · notebook 31 of 32 · 28 questions + a mini-project

Up to now your code has lived inside one cell. Real programs are run from a
terminal, read their settings from files and environment variables, keep
data in a database, call other programs, and ship as installable packages.
This notebook covers the standard-library tools for all of that.

Everything runs safely inside the notebook: command lines are passed to
`argparse` as explicit lists (in a notebook, `sys.argv` holds Jupyter's own
arguments), databases use SQLite's `":memory:"` mode, child processes are
started with `sys.executable` (the exact Python running this notebook),
and files live in temporary folders that are deleted afterwards.

Two traps run through the whole notebook: building SQL with f-strings
(SQL injection) and building shell commands as strings. You'll see both
go wrong, then learn the habits that make them impossible.

**Python 3.14:** the template-string (t-string) questions are tagged
`[3.14+]`. On an older Python, read them and skip running them.

## Prerequisites

Before you start, you should be comfortable with:

- [ ] NB13: files
- [ ] NB28: logging

## What you'll practise

- Project layout, packages, __main__.py
- argparse (called with an explicit argument list)
- Config from os.environ and tomllib (3.11+)
- sqlite3: in-memory databases, parameterised queries, transactions, row_factory
- subprocess.run with sys.executable
- shutil and zipfile
- venv, pip and pyproject.toml explained
- Template strings (t-strings, 3.14) for safe interpolation

## How to use this notebook

1. Read the question and write your code in the cell beneath it.
2. Run the cell, then open **Expected output** and compare.
3. Stuck? Open **Hint** first. Open **Solution** only after a real attempt.
4. Once it works, open **Senior dev solution** to see how an experienced developer would write it, and why.

The **Solution** only uses what this notebook and the ones before it teach. The **Senior dev solution** sometimes uses later topics on purpose, and says which notebook covers them.

| Type | What you do |
|---|---|
| **Write** | Write code that prints the answer |
| **Predict** | Read the code and predict exactly what it prints |
| **Error** | Predict which error the code raises |
| **Fix** | The code has a bug. Find it and fix it |
| **Refactor** | The code works. Make it cleaner without changing its output |

| Level | Meaning |
|---|---|
| L1 | Extremely Easy: one idea, a line or two |
| L2 | Easy: one idea with a small twist |
| L3 | Intermediate: two or three ideas combined |
| L4 | Somewhat Difficult: needs care to get exactly right |
| L5 | Hard: several ideas and a trap or two |

---

## L1 · Extremely Easy

### NB31-Q01 · L1 · sqlite3 · Write

Open an in-memory SQLite database with `sqlite3.connect(":memory:")`, run
the query `SELECT 2 + 3`, and print the row it returns.

**Starting code (already in the cell below):**

```python
import sqlite3
```

<details><summary><b>Hint</b></summary>

`conn.execute(...)` runs SQL and gives back a cursor. `.fetchone()` on the cursor returns the next row.

</details>

<details><summary><b>Expected output</b></summary>

```text
(5,)
```

</details>

<details><summary><b>Solution</b></summary>

```python
conn = sqlite3.connect(":memory:")
row = conn.execute("SELECT 2 + 3").fetchone()
print(row)
conn.close()
```

`":memory:"` makes a database that lives in RAM and disappears when the connection closes, which is ideal for practice and for tests. `fetchone()` always returns a row as a **tuple**, even when there's only one column, so you get `(5,)`, not `5`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from contextlib import closing

with closing(sqlite3.connect(":memory:")) as conn:
    row = conn.execute("SELECT 2 + 3").fetchone()
print(row)
```

`with conn:` on a sqlite3 connection does **not** close it: it commits or rolls back a transaction (Q13). Seniors wrap the connection in `contextlib.closing` (NB24) so it's closed even if a query raises.

</details>

In [ ]:
import sqlite3

# your code here


### NB31-Q02 · L1 · Config from os.environ · Write

Programs read settings from environment variables so the same code can
run on your laptop and on a server without edits. Print the value of
`SHOP_CURRENCY`, then the value of `SHOP_TIMEZONE`, falling back to
`"UTC"` when it isn't set.

```text
EUR
UTC
```

**Starting code (already in the cell below):**

```python
import os

os.environ["SHOP_CURRENCY"] = "EUR"
os.environ.pop("SHOP_TIMEZONE", None)  # make sure this one is not set
```

<details><summary><b>Hint</b></summary>

`os.environ` behaves like a dictionary. Which dictionary method gives a default when a key is missing?

</details>

<details><summary><b>Expected output</b></summary>

```text
EUR
UTC
```

</details>

<details><summary><b>Solution</b></summary>

```python
print(os.environ["SHOP_CURRENCY"])
print(os.environ.get("SHOP_TIMEZONE", "UTC"))
```

`os.environ` maps variable names to **strings**. Square brackets raise `KeyError` for a missing name, which is right for a setting the program can't run without. `.get(name, default)` suits optional settings.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
currency = os.environ["SHOP_CURRENCY"]        # required: fail at startup if missing
timezone = os.getenv("SHOP_TIMEZONE", "UTC")  # optional, with a default
print(currency)
print(timezone)
```

`os.getenv` is shorthand for `os.environ.get`. Seniors read all config in one place at startup and split required from optional, so a missing setting fails at launch instead of halfway through a nightly job. Every value is text: a port still needs `int()`.

</details>

In [ ]:
import os

os.environ["SHOP_CURRENCY"] = "EUR"
os.environ.pop("SHOP_TIMEZONE", None)  # make sure this one is not set

# your code here


### NB31-Q03 · L1 · Project layout and __main__ · Predict

Every module has a `__name__`. Predict the output.

**What does this print?**

```python
import json

print(__name__)
print(json.__name__)
```

<details><summary><b>Hint</b></summary>

The code you run directly gets a special name. An imported module is named after its file.

</details>

<details><summary><b>Expected output</b></summary>

```text
__main__
json
```

</details>

<details><summary><b>Solution</b></summary>

The module Python is running directly (a script, `python -m package`, or a notebook cell) is always called `__main__`. An imported module gets its own name, here `json`. That's what the familiar guard `if __name__ == "__main__":` tests: run this only when the file is run directly, not when it's imported.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def main():
    print("running the program")


if __name__ == "__main__":
    main()
```

A `main()` function plus the guard is the standard shape of a script. Importing the file (for example from a test, NB28) defines things but runs nothing; running it calls `main()`. Keep the top level of a module to imports, constants and definitions.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L2 · Easy

### NB31-Q04 · L2 · argparse · Write

Build an `ArgumentParser` (with `prog="report"`) that takes a positional
argument `path` and an optional `--limit`, an `int` that defaults to `10`.
Parse `argv` by passing it to `parse_args`, then print the path, the
limit, and the limit doubled.

```text
sales.csv
5
10
```

**Starting code (already in the cell below):**

```python
import argparse

argv = ["sales.csv", "--limit", "5"]
```

<details><summary><b>Hint</b></summary>

`add_argument("path")` makes a positional argument; a name starting with `--` makes an option. `type=int` converts the text for you.

</details>

<details><summary><b>Expected output</b></summary>

```text
sales.csv
5
10
```

</details>

<details><summary><b>Solution</b></summary>

```python
parser = argparse.ArgumentParser(prog="report")
parser.add_argument("path")
parser.add_argument("--limit", type=int, default=10)
args = parser.parse_args(argv)
print(args.path)
print(args.limit)
print(args.limit * 2)
```

Command-line arguments always arrive as text. `type=int` converts `"5"` to `5`, which is why doubling gives `10` and not `"55"`. Passing `argv` explicitly matters here: `parse_args()` with no list reads `sys.argv`, and inside Jupyter that holds the kernel's own arguments.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def build_parser():
    parser = argparse.ArgumentParser(prog="report", description="Summarise a sales file.")
    parser.add_argument("path", help="CSV file to read")
    parser.add_argument("--limit", type=int, default=10, help="rows to show (default: 10)")
    return parser


def main(argv=None):
    args = build_parser().parse_args(argv)
    print(args.path)
    print(args.limit)
    print(args.limit * 2)


main(argv)
```

`main(argv=None)` is the testable CLI pattern: in a real run `None` makes argparse read `sys.argv`, while tests pass a list. The `help=` strings cost nothing and turn into a free `--help` page.

</details>

In [ ]:
import argparse

argv = ["sales.csv", "--limit", "5"]

# your code here


### NB31-Q05 · L2 · argparse · Predict

Flags, defaults and repeated options. Predict the output.

**What does this print?**

```python
import argparse

parser = argparse.ArgumentParser(prog="backup")
parser.add_argument("--dry-run", action="store_true")
parser.add_argument("--keep", type=int, default=7)
parser.add_argument("--tag", action="append")

args = parser.parse_args(["--tag", "daily", "--tag", "db"])
print(args.dry_run, args.keep, args.tag)
args = parser.parse_args(["--dry-run", "--keep", "3"])
print(args.dry_run, args.keep, args.tag)
```

<details><summary><b>Hint</b></summary>

`--dry-run` becomes an attribute with an underscore. What is the default of an `append` option that never appears?

</details>

<details><summary><b>Expected output</b></summary>

```text
False 7 ['daily', 'db']
True 3 None
```

</details>

<details><summary><b>Solution</b></summary>

Dashes in option names become underscores, so `--dry-run` is `args.dry_run`. `store_true` defaults to `False` and becomes `True` when the flag is present. `append` collects every use into a list, but when the option never appears its default is `None`, **not** an empty list. Code that loops over `args.tag` would crash on the second run.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import argparse

parser = argparse.ArgumentParser(prog="backup")
parser.add_argument("--tag", action="append")
args = parser.parse_args([])
tags = args.tag or []
print(tags)
```

`args.tag or []` turns `None` into an empty list at the one place the value is read, so the rest of the code never has to check for `None`.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB31-Q06 · L2 · argparse · Error

The user asked for a format the tool doesn't support. Which error reaches your code?

**Which error does this raise?**

```python
import argparse

parser = argparse.ArgumentParser(prog="convert")
parser.add_argument("source")
parser.add_argument("--to", choices=["csv", "json"], required=True)

args = parser.parse_args(["data.xlsx", "--to", "xml"])
print("converting", args.source)
```

<details><summary><b>Hint</b></summary>

argparse is designed for terminals. What does a command-line tool do when it gets bad arguments?

</details>

<details><summary><b>Expected output</b></summary>

```text
SystemExit
```

</details>

<details><summary><b>Solution</b></summary>

`SystemExit`. argparse prints a usage message and the problem to **stderr**, then calls `sys.exit(2)`, which raises `SystemExit`. It inherits from `BaseException`, not `Exception`, so `except Exception` won't catch it. In a terminal that's what you want; in a notebook or a test you have to catch it yourself.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import argparse

parser = argparse.ArgumentParser(prog="convert")
parser.add_argument("source")
parser.add_argument("--to", choices=["csv", "json"], required=True)

try:
    args = parser.parse_args(["data.xlsx", "--to", "xml"])
except SystemExit as exc:
    print("bad arguments, exit code", exc.code)
```

Catching `SystemExit` is fine at the edge of a program, such as a test or a notebook. `ArgumentParser(exit_on_error=False)` raises `argparse.ArgumentError` for many mistakes instead, which suits programs that use argparse to parse commands typed into a running app.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB31-Q07 · L2 · sqlite3 · Write

Create a table `products` with columns `name` and `price`. Insert every row
of `products` with **one** `executemany` call using `?` placeholders. Then
print how many rows the table holds, and the most expensive product.

```text
3
('stapler', 7.99)
```

**Starting code (already in the cell below):**

```python
import sqlite3

products = [("pen", 1.20), ("notebook", 3.50), ("stapler", 7.99)]
```

<details><summary><b>Hint</b></summary>

`executemany(sql, rows)` runs the same statement once per tuple. `SELECT COUNT(*)` counts rows; `ORDER BY price DESC LIMIT 1` keeps the top one.

</details>

<details><summary><b>Expected output</b></summary>

```text
3
('stapler', 7.99)
```

</details>

<details><summary><b>Solution</b></summary>

```python
conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE products (name TEXT, price REAL)")
conn.executemany("INSERT INTO products VALUES (?, ?)", products)
print(conn.execute("SELECT COUNT(*) FROM products").fetchone()[0])
print(conn.execute("SELECT name, price FROM products ORDER BY price DESC LIMIT 1").fetchone())
conn.close()
```

Each `?` is a slot that SQLite fills with a value from the tuple. The values never become part of the SQL text, which is what makes this safe (Q19). `COUNT(*)` returns a one-column row, so `[0]` takes the number out of the tuple.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
conn = sqlite3.connect(":memory:")
conn.execute("""
    CREATE TABLE products (
        name  TEXT NOT NULL UNIQUE,
        price REAL NOT NULL CHECK (price >= 0)
    )
""")
with conn:
    conn.executemany("INSERT INTO products (name, price) VALUES (?, ?)", products)
(count,) = conn.execute("SELECT COUNT(*) FROM products").fetchone()
print(count)
print(conn.execute("SELECT name, price FROM products ORDER BY price DESC LIMIT 1").fetchone())
conn.close()
```

Constraints (`NOT NULL`, `UNIQUE`, `CHECK`) let the database reject bad data no matter which code path wrote it. Naming the columns in `INSERT` keeps it working when someone adds a column later, and `with conn:` commits the batch as one transaction.

</details>

In [ ]:
import sqlite3

products = [("pen", 1.20), ("notebook", 3.50), ("stapler", 7.99)]

# your code here


### NB31-Q08 · L2 · subprocess · Write

Run a second Python process that prints `6 * 7`. Use `subprocess.run`
with the list `[sys.executable, "-c", "print(6 * 7)"]`, capture its output
as text, and print the return code and the output without its trailing
newline.

```text
0
42
```

**Starting code (already in the cell below):**

```python
import subprocess
import sys
```

<details><summary><b>Hint</b></summary>

`capture_output=True` collects what the child prints; `text=True` gives you `str` instead of `bytes`. The result object has `.returncode` and `.stdout`.

</details>

<details><summary><b>Expected output</b></summary>

```text
0
42
```

</details>

<details><summary><b>Solution</b></summary>

```python
result = subprocess.run([sys.executable, "-c", "print(6 * 7)"], capture_output=True, text=True)
print(result.returncode)
print(result.stdout.strip())
```

`sys.executable` is the full path of the Python running this notebook. Typing `"python"` instead might start a different version, or nothing at all on Windows. Each list item is one argument, passed as-is. A return code of `0` means success; the child's `print` adds a newline, which `.strip()` removes.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
result = subprocess.run(
    [sys.executable, "-c", "print(6 * 7)"],
    capture_output=True, text=True, check=True, timeout=10,
)
print(result.returncode)
print(result.stdout.strip())
```

`check=True` turns a failing child into an exception (Q14), so a failure can't slip by unnoticed. `timeout=` stops a stuck child from freezing your program forever. Seniors add both by default.

</details>

In [ ]:
import subprocess
import sys

# your code here


### NB31-Q09 · L2 · Config from tomllib · Write

Parse the TOML text with `tomllib.loads` and print the host, the port plus
one, the type name of `debug`, and the database path.

```text
127.0.0.1
8081
bool
shop.db
```

**Starting code (already in the cell below):**

```python
import tomllib

text = """
[server]
host = "127.0.0.1"
port = 8080
debug = false

[database]
path = "shop.db"
"""
```

<details><summary><b>Hint</b></summary>

`tomllib.loads` returns nested dictionaries: one per `[table]`. `type(value).__name__` gives a type's name as text.

</details>

<details><summary><b>Expected output</b></summary>

```text
127.0.0.1
8081
bool
shop.db
```

</details>

<details><summary><b>Solution</b></summary>

```python
config = tomllib.loads(text)
print(config["server"]["host"])
print(config["server"]["port"] + 1)
print(type(config["server"]["debug"]).__name__)
print(config["database"]["path"])
```

Unlike environment variables (always text) or INI files (NB13), TOML values have real types: `8080` arrives as an `int` and `false` as a `bool`, so no conversion is needed. Each `[table]` becomes a nested dict. `tomllib` has been in the standard library since Python 3.11, and it only reads TOML; it can't write it.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
config = tomllib.loads(text)
server = config["server"]
print(server["host"])
print(server["port"] + 1)
print(type(server["debug"]).__name__)
print(config.get("database", {}).get("path", "app.db"))
```

Name a sub-table once instead of repeating `config["server"]`, and use `.get` with defaults for optional sections. To read a real file, open it in **binary** mode: `with open(path, "rb") as f: tomllib.load(f)`. Text mode raises `TypeError`.

</details>

In [ ]:
import tomllib

text = """
[server]
host = "127.0.0.1"
port = 8080
debug = false

[database]
path = "shop.db"
"""

# your code here


---

## L3 · Intermediate

### NB31-Q10 · L3 · argparse · Write

Build a CLI called `budget` with two subcommands, using
`add_subparsers(dest="command", required=True)`:

- `add` takes `amount` (a `float`) and `category`, plus an optional
  `--note` that defaults to `"-"`.
- `list` takes an optional `--month` that defaults to `"all"`.

Parse each argv list in `runs` and print one line per run:

```text
add 12.5 food -
add 3.0 transport bus
list 2026-09
```

**Starting code (already in the cell below):**

```python
import argparse

runs = [
    ["add", "12.50", "food"],
    ["add", "3", "transport", "--note", "bus"],
    ["list", "--month", "2026-09"],
]
```

<details><summary><b>Hint</b></summary>

`add_subparsers` returns an object whose `.add_parser("add")` gives you a parser for that subcommand. `args.command` tells you which one was used.

</details>

<details><summary><b>Expected output</b></summary>

```text
add 12.5 food -
add 3.0 transport bus
list 2026-09
```

</details>

<details><summary><b>Solution</b></summary>

```python
parser = argparse.ArgumentParser(prog="budget")
sub = parser.add_subparsers(dest="command", required=True)

add = sub.add_parser("add")
add.add_argument("amount", type=float)
add.add_argument("category")
add.add_argument("--note", default="-")

lst = sub.add_parser("list")
lst.add_argument("--month", default="all")

for argv in runs:
    args = parser.parse_args(argv)
    if args.command == "add":
        print(args.command, args.amount, args.category, args.note)
    else:
        print(args.command, args.month)
```

Each subcommand has its own parser with its own arguments, just like `git commit` and `git push`. `dest="command"` stores which subcommand was chosen, and `required=True` makes running `budget` with no subcommand an error. `"12.50"` becomes the float `12.5`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def do_add(args):
    print("add", args.amount, args.category, args.note)


def do_list(args):
    print("list", args.month)


parser = argparse.ArgumentParser(prog="budget")
sub = parser.add_subparsers(dest="command", required=True)

add = sub.add_parser("add", help="record an expense")
add.add_argument("amount", type=float)
add.add_argument("category")
add.add_argument("--note", default="-")
add.set_defaults(handler=do_add)

lst = sub.add_parser("list", help="show expenses")
lst.add_argument("--month", default="all")
lst.set_defaults(handler=do_list)

for argv in runs:
    args = parser.parse_args(argv)
    args.handler(args)
```

`set_defaults(handler=...)` attaches a function to each subcommand, so dispatch is one line and adding a third command never touches an `if`/`elif` chain. Large CLIs like pip are organised this way.

</details>

In [ ]:
import argparse

runs = [
    ["add", "12.50", "food"],
    ["add", "3", "transport", "--note", "bus"],
    ["list", "--month", "2026-09"],
]

# your code here


### NB31-Q11 · L3 · sqlite3 · Write

Create, read, update, delete. Use `?` placeholders for every value:

1. Insert `("apples", 10)`, `("pears", 4)` and `("plums", 0)`.
2. Add `5` to the apples' quantity.
3. Delete every item whose quantity is `0`.
4. Commit, then print every remaining row, ordered by item.

```text
('apples', 15)
('pears', 4)
```

**Starting code (already in the cell below):**

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE stock (item TEXT PRIMARY KEY, qty INTEGER)")
```

<details><summary><b>Hint</b></summary>

`UPDATE stock SET qty = qty + ? WHERE item = ?` adds inside the database. The parameters must always be a tuple, even for one value: `(0,)`.

</details>

<details><summary><b>Expected output</b></summary>

```text
('apples', 15)
('pears', 4)
```

</details>

<details><summary><b>Solution</b></summary>

```python
for item, qty in [("apples", 10), ("pears", 4), ("plums", 0)]:
    conn.execute("INSERT INTO stock (item, qty) VALUES (?, ?)", (item, qty))
conn.execute("UPDATE stock SET qty = qty + ? WHERE item = ?", (5, "apples"))
conn.execute("DELETE FROM stock WHERE qty = ?", (0,))
conn.commit()
for row in conn.execute("SELECT item, qty FROM stock ORDER BY item"):
    print(row)
```

`qty = qty + ?` does the arithmetic inside the database, so you never read the old value into Python and write it back (two programs doing that at once would lose an update). `(0,)` is a one-item tuple; `(0)` is just the number `0` and raises an error. A cursor can be looped over directly, one row at a time.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
with conn:
    conn.executemany(
        "INSERT INTO stock (item, qty) VALUES (:item, :qty)",
        [{"item": "apples", "qty": 10}, {"item": "pears", "qty": 4}, {"item": "plums", "qty": 0}],
    )
    conn.execute("UPDATE stock SET qty = qty + :extra WHERE item = :item",
                 {"extra": 5, "item": "apples"})
    conn.execute("DELETE FROM stock WHERE qty = 0")
for row in conn.execute("SELECT item, qty FROM stock ORDER BY item"):
    print(row)
```

Named placeholders (`:item`) with a dict read better once a query has more than two values. A constant you wrote yourself, like the `0` in the `DELETE`, is safe to leave in the SQL; only values from outside the program must go through placeholders.

</details>

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE stock (item TEXT PRIMARY KEY, qty INTEGER)")

# your code here


### NB31-Q12 · L3 · sqlite3 row_factory · Write

Set `conn.row_factory = sqlite3.Row` so rows can be read by column name.
Query everyone on the `data` team, highest salary first, and print each as
`name: salary`. Then print the column names of the last row as a list, and
the last row as a dict.

```text
Grace: 6100
Ada: 5200
['id', 'name', 'team', 'salary']
{'id': 1, 'name': 'Ada', 'team': 'data', 'salary': 5200}
```

**Starting code (already in the cell below):**

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE staff (id INTEGER PRIMARY KEY, name TEXT, team TEXT, salary INTEGER)")
conn.executemany("INSERT INTO staff (name, team, salary) VALUES (?, ?, ?)",
                 [("Ada", "data", 5200), ("Linus", "infra", 4800), ("Grace", "data", 6100)])
```

<details><summary><b>Hint</b></summary>

With `sqlite3.Row`, `row["name"]` works. A `Row` has a `.keys()` method, and `dict()` accepts anything that has `.keys()` and indexing.

</details>

<details><summary><b>Expected output</b></summary>

```text
Grace: 6100
Ada: 5200
['id', 'name', 'team', 'salary']
{'id': 1, 'name': 'Ada', 'team': 'data', 'salary': 5200}
```

</details>

<details><summary><b>Solution</b></summary>

```python
conn.row_factory = sqlite3.Row
rows = conn.execute("SELECT * FROM staff WHERE team = ? ORDER BY salary DESC", ("data",)).fetchall()
for row in rows:
    print(f"{row['name']}: {row['salary']}")
print(rows[-1].keys())
print(dict(rows[-1]))
```

By default rows are plain tuples, so you'd write `row[3]` and have to remember that column 3 is the salary. `sqlite3.Row` lets you index by column name (and still by position). `dict(row)` is handy when the data has to become JSON.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
conn.row_factory = sqlite3.Row
rows = conn.execute(
    "SELECT id, name, team, salary FROM staff WHERE team = :team ORDER BY salary DESC",
    {"team": "data"},
).fetchall()
for row in rows:
    print(f"{row['name']}: {row['salary']}")
print(rows[-1].keys())
print(dict(rows[-1]))
```

Seniors list the columns instead of `SELECT *`. The query then documents exactly what the code depends on, and adding a column to the table can't silently change the output.

</details>

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE staff (id INTEGER PRIMARY KEY, name TEXT, team TEXT, salary INTEGER)")
conn.executemany("INSERT INTO staff (name, team, salary) VALUES (?, ?, ?)",
                 [("Ada", "data", 5200), ("Linus", "infra", 4800), ("Grace", "data", 6100)])

# your code here


### NB31-Q13 · L3 · sqlite3 transactions · Predict

A transfer that fails halfway. Predict the output.

**What does this print?**

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER)")

with conn:
    conn.execute("INSERT INTO accounts VALUES ('ana', 100)")
    conn.execute("INSERT INTO accounts VALUES ('ben', 50)")

try:
    with conn:
        conn.execute("UPDATE accounts SET balance = balance - 30 WHERE name = 'ana'")
        conn.execute("INSERT INTO accounts VALUES ('ben', 0)")
except sqlite3.IntegrityError:
    print("transfer failed")

print(conn.execute("SELECT name, balance FROM accounts ORDER BY name").fetchall())
```

<details><summary><b>Hint</b></summary>

`name` is the primary key, so a second `'ben'` is rejected. What does `with conn:` do when the block raises?

</details>

<details><summary><b>Expected output</b></summary>

```text
transfer failed
[('ana', 100), ('ben', 50)]
```

</details>

<details><summary><b>Solution</b></summary>

`with conn:` wraps the block in a transaction: it **commits** if the block finishes and **rolls back** if it raises. The `UPDATE` took 30 from ana, but the duplicate `INSERT` raised `IntegrityError`, so the whole block was undone and ana still has 100. Either every change in a transaction happens, or none does.

</details>

<details><summary><b>Senior dev solution</b></summary>

This all-or-nothing behaviour is why money code always groups related writes in one transaction (Q20). Remember that `with conn:` doesn't close the connection, it only ends the transaction.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

### NB31-Q14 · L3 · subprocess · Write

The `child` program fails with exit code 3. Run it with `check=True`,
capturing text output, catch `subprocess.CalledProcessError`, and print
the exit code and what the child wrote to stdout and to stderr:

```text
failed with code 3
stdout: checking disk
stderr: disk full
```

**Starting code (already in the cell below):**

```python
import subprocess
import sys

child = "import sys; print('checking disk'); print('disk full', file=sys.stderr); sys.exit(3)"
```

<details><summary><b>Hint</b></summary>

The exception object carries `.returncode`, `.stdout` and `.stderr`.

</details>

<details><summary><b>Expected output</b></summary>

```text
failed with code 3
stdout: checking disk
stderr: disk full
```

</details>

<details><summary><b>Solution</b></summary>

```python
try:
    subprocess.run([sys.executable, "-c", child], capture_output=True, text=True, check=True)
except subprocess.CalledProcessError as err:
    print("failed with code", err.returncode)
    print("stdout:", err.stdout.strip())
    print("stderr:", err.stderr.strip())
```

Programs report failure through their **exit code**: `0` means success, anything else means failure. Without `check=True`, `subprocess.run` returns normally even when the child fails. With it, a non-zero code raises `CalledProcessError`, which still holds everything the child printed. stdout and stderr are captured separately.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
result = subprocess.run([sys.executable, "-c", child], capture_output=True, text=True, timeout=10)
if result.returncode != 0:
    print("failed with code", result.returncode)
    print("stdout:", result.stdout.strip())
    print("stderr:", result.stderr.strip())
```

Both styles are correct. Use `check=True` when failure is exceptional; check `returncode` yourself when some non-zero codes are expected (for example, `grep` returns `1` when nothing matches). The only wrong choice is ignoring it (Q27).

</details>

In [ ]:
import subprocess
import sys

child = "import sys; print('checking disk'); print('disk full', file=sys.stderr); sys.exit(3)"

# your code here


### NB31-Q15 · L3 · shutil · Refactor

This copies a folder tree by hand: walking it, making folders, reading and
writing every file. Replace the copying loop with one call from `shutil`.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "photos" / "2026").mkdir(parents=True)
    (root / "photos" / "cover.jpg").write_bytes(b"JPEG cover")
    (root / "photos" / "2026" / "beach.jpg").write_bytes(b"JPEG beach")

    backup = root / "backup"
    for src in (root / "photos").rglob("*"):
        dest = backup / src.relative_to(root / "photos")
        if src.is_dir():
            dest.mkdir(parents=True, exist_ok=True)
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(src, "rb") as f_in:
                data = f_in.read()
            with open(dest, "wb") as f_out:
                f_out.write(data)

    for path in sorted(backup.rglob("*")):
        print(path.relative_to(root).as_posix())
```

<details><summary><b>Hint</b></summary>

`shutil` has a function that copies a whole directory tree, creating the destination for you.

</details>

<details><summary><b>Expected output</b></summary>

```text
backup/2026
backup/2026/beach.jpg
backup/cover.jpg
```

</details>

<details><summary><b>Solution</b></summary>

```python
import shutil
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "photos" / "2026").mkdir(parents=True)
    (root / "photos" / "cover.jpg").write_bytes(b"JPEG cover")
    (root / "photos" / "2026" / "beach.jpg").write_bytes(b"JPEG beach")

    backup = root / "backup"
    shutil.copytree(root / "photos", backup)

    for path in sorted(backup.rglob("*")):
        print(path.relative_to(root).as_posix())
```

`shutil.copytree(src, dest)` creates `dest` and copies every folder and file under `src`. It also copies file metadata such as modification times, which the hand-written loop lost, and it streams large files in chunks instead of reading each one fully into memory.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import shutil
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "photos" / "2026").mkdir(parents=True)
    (root / "photos" / "cover.jpg").write_bytes(b"JPEG cover")
    (root / "photos" / "2026" / "beach.jpg").write_bytes(b"JPEG beach")

    backup = root / "backup"
    shutil.copytree(root / "photos", backup, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("*.tmp", "__pycache__"))

    for path in sorted(backup.rglob("*")):
        print(path.relative_to(root).as_posix())
```

`dirs_exist_ok=True` lets a backup run again into the same folder, and `ignore=` skips junk files. `shutil` also has `move`, `rmtree`, `make_archive` and `which`. Python 3.14 adds `Path.copy()` for single files and trees.

</details>

In [ ]:
import tempfile
from pathlib import Path

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    (root / "photos" / "2026").mkdir(parents=True)
    (root / "photos" / "cover.jpg").write_bytes(b"JPEG cover")
    (root / "photos" / "2026" / "beach.jpg").write_bytes(b"JPEG beach")

    backup = root / "backup"
    for src in (root / "photos").rglob("*"):
        dest = backup / src.relative_to(root / "photos")
        if src.is_dir():
            dest.mkdir(parents=True, exist_ok=True)
        else:
            dest.parent.mkdir(parents=True, exist_ok=True)
            with open(src, "rb") as f_in:
                data = f_in.read()
            with open(dest, "wb") as f_out:
                f_out.write(data)

    for path in sorted(backup.rglob("*")):
        print(path.relative_to(root).as_posix())


### NB31-Q16 · L3 · zipfile · Write

Create `archive.zip` in `root` containing both text files, stored under
their plain file names and compressed with `zipfile.ZIP_DEFLATED`. Open the
archive again for reading, print its sorted member names, then print the
contents of `notes.txt` read straight from the archive. Finish with
`workdir.cleanup()`.

```text
['notes.txt', 'todo.txt']
buy milk
call bank
```

**Starting code (already in the cell below):**

```python
import tempfile
import zipfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
(root / "notes.txt").write_text("buy milk\ncall bank\n")
(root / "todo.txt").write_text("fix bike\n")
```

<details><summary><b>Hint</b></summary>

`ZipFile(path, "w", compression=...)` opens a new archive. `.write(file, arcname=...)` controls the name stored inside. `.read(name)` returns **bytes**.

</details>

<details><summary><b>Expected output</b></summary>

```text
['notes.txt', 'todo.txt']
buy milk
call bank
```

</details>

<details><summary><b>Solution</b></summary>

```python
archive = root / "archive.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(root / "notes.txt", arcname="notes.txt")
    zf.write(root / "todo.txt", arcname="todo.txt")

with zipfile.ZipFile(archive) as zf:
    print(sorted(zf.namelist()))
    print(zf.read("notes.txt").decode("utf-8"), end="")

workdir.cleanup()
```

Without `arcname`, the zip would store the file under its full temporary path, folders and all. `ZIP_DEFLATED` actually compresses (the default, `ZIP_STORED`, doesn't). `zf.read` returns bytes, so `.decode` turns them into text; `end=""` avoids a doubled final newline.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
archive = root / "archive.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(root.glob("*.txt")):
        zf.write(path, arcname=path.name)

with zipfile.ZipFile(archive) as zf:
    print(sorted(zf.namelist()))
    with zf.open("notes.txt") as member:
        print(member.read().decode("utf-8"), end="")

workdir.cleanup()
```

A loop over `glob` scales to any number of files, and `zf.open` gives a file-like object you can stream from. Seniors are careful extracting archives from strangers: member names like `../../.bashrc` can escape the target folder. `zipfile` cleans those names; for `tarfile`, pass `filter="data"`.

</details>

In [ ]:
import tempfile
import zipfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
(root / "notes.txt").write_text("buy milk\ncall bank\n")
(root / "todo.txt").write_text("fix bike\n")

# your code here


### NB31-Q17 · L3 · Shell command strings · Predict

`shlex.split` splits a command line the way a Unix shell does. Here a
command is built by gluing strings together. Predict the output.

**What does this print?**

```python
import shlex

filename = "holiday photo.jpg"
command = "convert " + filename + " small.png"
print(shlex.split(command))

filename = "x.jpg; rm -rf ~"
command = "convert " + filename + " small.png"
print(shlex.split(command))
```

<details><summary><b>Hint</b></summary>

To a shell, a space separates arguments, whatever you meant by it.

</details>

<details><summary><b>Expected output</b></summary>

```text
['convert', 'holiday', 'photo.jpg', 'small.png']
['convert', 'x.jpg;', 'rm', '-rf', '~', 'small.png']
```

</details>

<details><summary><b>Solution</b></summary>

The first file name contains a space, so the shell sees **two** arguments, `holiday` and `photo.jpg`, and `convert` looks for files that don't exist. The second is worse. `shlex.split` keeps `x.jpg;` as a word, but a real shell treats `;` as "end of command" and would run `rm -rf ~` next, deleting your home folder. Building a command string out of data is the shell version of SQL injection.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import shlex

filename = "holiday photo.jpg"
args = ["convert", filename, "small.png"]   # a list: no splitting, no shell involved
print(args)
print(shlex.join(args))                     # only for showing or logging the command
```

Pass `subprocess.run` a **list** and never use `shell=True` with data in it. Each item reaches the program as exactly one argument, whatever characters it contains. `shlex.join` quotes a list safely when you need to show the command to a person.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L4 · Somewhat Difficult

### NB31-Q18 · L4 · SQL injection · Error

A customer's name is pasted into the SQL with an f-string. Predict what
is printed and which error is raised.

**Which error does this raise?**

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE customers (name TEXT, city TEXT)")
conn.execute("INSERT INTO customers VALUES ('Siobhan', 'Cork')")

name = "Siobhan O'Brien"
query = f"SELECT city FROM customers WHERE name = '{name}'"
print(query)
print(conn.execute(query).fetchall())
```

<details><summary><b>Hint</b></summary>

Look at the printed query. Where does the quoted text end, as far as SQLite can tell?

</details>

<details><summary><b>Expected output</b></summary>

```text
SELECT city FROM customers WHERE name = 'Siobhan O'Brien'
OperationalError
```

</details>

<details><summary><b>Solution</b></summary>

`OperationalError`. The apostrophe in `O'Brien` closes the SQL string early, leaving `Brien'` as garbage, so SQLite reports a syntax error. An honest customer just crashed your program. Someone who types a quote **on purpose** can do much worse (Q19), because whatever follows the quote is run as SQL.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE customers (name TEXT, city TEXT)")
conn.execute("INSERT INTO customers VALUES ('Siobhan', 'Cork')")

name = "Siobhan O'Brien"
print(conn.execute("SELECT city FROM customers WHERE name = ?", (name,)).fetchall())
```

With a `?` placeholder the value travels separately from the SQL, so any character in it is just data. The query runs and correctly finds no match. Escaping quotes by hand is not the fix: it's easy to miss a case.

</details>

In [ ]:
# Which error? Write its name here, then open Expected output:
# 

### NB31-Q19 · L4 · SQL injection · Fix

The second call is an attacker typing a crafted "username" into a login
form, and it leaks **every** API key. Fix `find_key` so nothing a user
types can change the shape of the query. After the fix the output is:

```text
[('ada', 'key-ada-111')]
[]
```

**Buggy code (copied into the cell below):**

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (username TEXT, api_key TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?)",
                 [("ada", "key-ada-111"), ("bob", "key-bob-222")])


def find_key(username):
    query = f"SELECT username, api_key FROM users WHERE username = '{username}'"
    return conn.execute(query).fetchall()


print(find_key("ada"))
print(find_key("nobody' OR '1'='1"))
```

<details><summary><b>Hint</b></summary>

Write out the SQL the f-string builds for the second call. Then replace the f-string with a placeholder.

</details>

<details><summary><b>Expected output</b></summary>

```text
[('ada', 'key-ada-111')]
[]
```

</details>

<details><summary><b>Solution</b></summary>

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (username TEXT, api_key TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?)",
                 [("ada", "key-ada-111"), ("bob", "key-bob-222")])


def find_key(username):
    query = "SELECT username, api_key FROM users WHERE username = ?"
    return conn.execute(query, (username,)).fetchall()


print(find_key("ada"))
print(find_key("nobody' OR '1'='1"))
```

The f-string built `... WHERE username = 'nobody' OR '1'='1'`. Since `'1'='1'` is true for every row, the query returned the whole table. With `?`, SQLite compares the username column with the literal text `nobody' OR '1'='1`, which nobody has, so the result is empty.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (username TEXT, api_key TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?)",
                 [("ada", "key-ada-111"), ("bob", "key-bob-222")])

FIND_KEY_SQL = "SELECT username, api_key FROM users WHERE username = :username"


def find_key(username):
    return conn.execute(FIND_KEY_SQL, {"username": username}).fetchall()


print(find_key("ada"))
print(find_key("nobody' OR '1'='1"))
```

Keeping the SQL in a constant makes it obvious that it never changes; only the parameters do. A real system would also store a **hash** of each key (`hashlib`) instead of the key itself, and compare with `hmac.compare_digest`, so a leak like this one exposes nothing usable.

</details>

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE users (username TEXT, api_key TEXT)")
conn.executemany("INSERT INTO users VALUES (?, ?)",
                 [("ada", "key-ada-111"), ("bob", "key-bob-222")])


def find_key(username):
    query = f"SELECT username, api_key FROM users WHERE username = '{username}'"
    return conn.execute(query).fetchall()


print(find_key("ada"))
print(find_key("nobody' OR '1'='1"))


### NB31-Q20 · L4 · sqlite3 transactions · Fix

`transfer` moves money between accounts. A balance can't go negative (the
`CHECK` constraint), so the second transfer is refused, but afterwards ana
has 500 more than she should. Money was created from nothing. Fix
`transfer` so a refused transfer changes nothing.

**Buggy code (copied into the cell below):**

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("ana", 100), ("ben", 20)])
conn.commit()


def transfer(source, target, amount):
    try:
        conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, target))
        conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, source))
        conn.commit()
        print("ok:", source, "->", target, amount)
    except sqlite3.IntegrityError:
        print("refused:", source, "->", target, amount)


transfer("ana", "ben", 30)
transfer("ben", "ana", 500)
print(conn.execute("SELECT name, balance FROM accounts ORDER BY name").fetchall())
```

<details><summary><b>Hint</b></summary>

The first `UPDATE` succeeds before the second one fails. Nothing ever undoes it. Which tool from Q13 undoes everything when a block raises?

</details>

<details><summary><b>Expected output</b></summary>

```text
ok: ana -> ben 30
refused: ben -> ana 500
[('ana', 70), ('ben', 50)]
```

</details>

<details><summary><b>Solution</b></summary>

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("ana", 100), ("ben", 20)])
conn.commit()


def transfer(source, target, amount):
    try:
        with conn:
            conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, target))
            conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, source))
        print("ok:", source, "->", target, amount)
    except sqlite3.IntegrityError:
        print("refused:", source, "->", target, amount)


transfer("ana", "ben", 30)
transfer("ben", "ana", 500)
print(conn.execute("SELECT name, balance FROM accounts ORDER BY name").fetchall())
```

A failed statement only undoes **itself**. The credit to ana had already happened and stayed in the open transaction, which the catch block never rolled back. Wrapping both updates in `with conn:` makes them one unit: the `IntegrityError` triggers a rollback of the credit too, and only then reaches the `except`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("ana", 100), ("ben", 20)])
conn.commit()

DEBIT = "UPDATE accounts SET balance = balance - ? WHERE name = ?"
CREDIT = "UPDATE accounts SET balance = balance + ? WHERE name = ?"


def transfer(source, target, amount):
    try:
        with conn:
            for sql, name in ((DEBIT, source), (CREDIT, target)):
                if conn.execute(sql, (amount, name)).rowcount != 1:
                    raise LookupError(f"no account {name!r}")
    except (sqlite3.IntegrityError, LookupError):
        print("refused:", source, "->", target, amount)
    else:
        print("ok:", source, "->", target, amount)


transfer("ana", "ben", 30)
transfer("ben", "ana", 500)
print(conn.execute("SELECT name, balance FROM accounts ORDER BY name").fetchall())
```

The senior debits first, so the most likely failure happens before any money moves, and checks `rowcount`: an `UPDATE` that matches no row (a typo in an account name) is **not** an error in SQL and would otherwise succeed silently. `try`/`except`/`else` (NB12) keeps the success message out of the guarded block.

</details>

In [ ]:
import sqlite3

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE accounts (name TEXT PRIMARY KEY, balance INTEGER CHECK (balance >= 0))")
conn.executemany("INSERT INTO accounts VALUES (?, ?)", [("ana", 100), ("ben", 20)])
conn.commit()


def transfer(source, target, amount):
    try:
        conn.execute("UPDATE accounts SET balance = balance + ? WHERE name = ?", (amount, target))
        conn.execute("UPDATE accounts SET balance = balance - ? WHERE name = ?", (amount, source))
        conn.commit()
        print("ok:", source, "->", target, amount)
    except sqlite3.IntegrityError:
        print("refused:", source, "->", target, amount)


transfer("ana", "ben", 30)
transfer("ben", "ana", 500)
print(conn.execute("SELECT name, balance FROM accounts ORDER BY name").fetchall())


### NB31-Q21 · L4 · Project layout and __main__ · Write

The folder `inventory/` is a **package**: it has an `__init__.py`. Write
`inventory/__main__.py` so that `python -m inventory <item>` prints the
package version and the stock count. It should import `VERSION` from the
package, import `count` from `inventory.core`, and read the item from
`sys.argv[1]`.

Then run it for `apples` and for `kiwis` with
`subprocess.run([sys.executable, "-m", "inventory", item], cwd=root, ...)`,
print what each run printed, and clean up.

```text
inventory 1.2.0
apples: 12
inventory 1.2.0
kiwis: 0
```

**Starting code (already in the cell below):**

```python
import subprocess
import sys
import tempfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
package = root / "inventory"
package.mkdir()
(package / "__init__.py").write_text('VERSION = "1.2.0"\n')
(package / "core.py").write_text(
    'STOCK = {"apples": 12, "pears": 0}\n'
    "\n"
    "\n"
    "def count(item):\n"
    "    return STOCK.get(item, 0)\n"
)
```

<details><summary><b>Hint</b></summary>

`python -m inventory` runs the file `inventory/__main__.py`. Write its source with `write_text`, one line per string. `cwd=root` lets the child find the package.

</details>

<details><summary><b>Expected output</b></summary>

```text
inventory 1.2.0
apples: 12
inventory 1.2.0
kiwis: 0
```

</details>

<details><summary><b>Solution</b></summary>

```python
(package / "__main__.py").write_text(
    "import sys\n"
    "from inventory import VERSION\n"
    "from inventory.core import count\n"
    "\n"
    'print("inventory", VERSION)\n'
    "item = sys.argv[1]\n"
    'print(f"{item}: {count(item)}")\n'
)

for item in ["apples", "kiwis"]:
    result = subprocess.run([sys.executable, "-m", "inventory", item],
                            cwd=root, capture_output=True, text=True, check=True)
    print(result.stdout, end="")

workdir.cleanup()
```

A folder with `__init__.py` is a package, and `__init__.py` runs on the first import (here it defines `VERSION`). `python -m inventory` imports the package and then runs its `__main__.py` as the main module. `-m` puts the current directory on the import path, which is why `cwd=root` matters. The typical layout is: `__init__.py` for the public names, modules like `core.py` for the logic, and a thin `__main__.py` for the command line.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import textwrap

(package / "__main__.py").write_text(textwrap.dedent('''
    import argparse

    from . import VERSION
    from .core import count


    def main(argv=None):
        parser = argparse.ArgumentParser(prog="inventory")
        parser.add_argument("item")
        args = parser.parse_args(argv)
        print("inventory", VERSION)
        print(f"{args.item}: {count(args.item)}")


    if __name__ == "__main__":
        main()
'''))

for item in ["apples", "kiwis"]:
    result = subprocess.run([sys.executable, "-m", "inventory", item],
                            cwd=root, capture_output=True, text=True, check=True, timeout=30)
    print(result.stdout, end="")

workdir.cleanup()
```

Inside a package, seniors use relative imports (`from .core import count`) so the package still works if it's renamed. `__main__.py` holds a testable `main(argv=None)` and the `__name__` guard (Q03). Real projects usually put the package under a `src/` folder, next to `pyproject.toml` and `tests/`, so tests run against the installed package.

</details>

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
package = root / "inventory"
package.mkdir()
(package / "__init__.py").write_text('VERSION = "1.2.0"\n')
(package / "core.py").write_text(
    'STOCK = {"apples": 12, "pears": 0}\n'
    "\n"
    "\n"
    "def count(item):\n"
    "    return STOCK.get(item, 0)\n"
)

# your code here


### NB31-Q22 · L4 · argparse · Refactor

This parses its options by hand with a `while` loop. Rewrite it with
`argparse`, keeping the output identical.

**Working code to refactor (copied into the cell below). Keep the output exactly the same:**

```python
argv = ["--name", "Ada", "--count", "3", "--loud"]

name = "world"
count = 1
loud = False
i = 0
while i < len(argv):
    if argv[i] == "--name":
        name = argv[i + 1]
        i += 2
    elif argv[i] == "--count":
        count = int(argv[i + 1])
        i += 2
    elif argv[i] == "--loud":
        loud = True
        i += 1
    else:
        i += 1

greeting = "hello " + name
if loud:
    greeting = greeting.upper() + "!"
for _ in range(count):
    print(greeting)
```

<details><summary><b>Hint</b></summary>

Three options: one string with a default, one `int` with a default, and one on/off flag (`action="store_true"`).

</details>

<details><summary><b>Expected output</b></summary>

```text
HELLO ADA!
HELLO ADA!
HELLO ADA!
```

</details>

<details><summary><b>Solution</b></summary>

```python
import argparse

argv = ["--name", "Ada", "--count", "3", "--loud"]

parser = argparse.ArgumentParser(prog="greet")
parser.add_argument("--name", default="world")
parser.add_argument("--count", type=int, default=1)
parser.add_argument("--loud", action="store_true")
args = parser.parse_args(argv)

greeting = "hello " + args.name
if args.loud:
    greeting = greeting.upper() + "!"
for _ in range(args.count):
    print(greeting)
```

The hand-written loop has hidden bugs: a typo like `--cuont` is silently ignored, and `--name` as the last word raises `IndexError`. argparse rejects unknown options, reports missing values clearly, accepts `--count=3` as well as `--count 3`, and generates `--help` for free.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import argparse

argv = ["--name", "Ada", "--count", "3", "--loud"]


def positive_int(text):
    value = int(text)
    if value < 1:
        raise argparse.ArgumentTypeError(f"must be at least 1, got {value}")
    return value


parser = argparse.ArgumentParser(prog="greet", description="Say hello.")
parser.add_argument("-n", "--name", default="world", help="who to greet")
parser.add_argument("-c", "--count", type=positive_int, default=1, help="how many times")
parser.add_argument("--loud", action="store_true", help="shout")
args = parser.parse_args(argv)

greeting = f"hello {args.name}"
if args.loud:
    greeting = f"{greeting.upper()}!"
print("\n".join([greeting] * args.count))
```

`type=` accepts any function, so validation (here: at least 1) happens while parsing, with a proper error message. Short aliases (`-n`) and `help=` text make the tool pleasant to use.

</details>

In [ ]:
argv = ["--name", "Ada", "--count", "3", "--loud"]

name = "world"
count = 1
loud = False
i = 0
while i < len(argv):
    if argv[i] == "--name":
        name = argv[i + 1]
        i += 2
    elif argv[i] == "--count":
        count = int(argv[i + 1])
        i += 2
    elif argv[i] == "--loud":
        loud = True
        i += 1
    else:
        i += 1

greeting = "hello " + name
if loud:
    greeting = greeting.upper() + "!"
for _ in range(count):
    print(greeting)


### NB31-Q23 · L4 · venv, pip and pyproject.toml · Write

When you run `pip install .`, pip reads `pyproject.toml`, installs the
`dependencies`, and turns each `[project.scripts]` entry into a command
you can type. Do a small version of that by hand:

1. Parse `pyproject` and print `name version (python >=3.12)`.
2. Print the package name of each dependency: the part before any `<`,
   `>`, `=`, `!` or `~`.
3. For each script, print `command -> target`. Split the target
   `"module:function"`, put `root` at the front of `sys.path`, import the
   module with `importlib.import_module`, get the function with
   `getattr`, and call it.

Then remove `root` from `sys.path` and clean up.

```text
tally 0.3.1 (python >=3.12)
rich
httpx
tally -> tally.cli:main
tally is running
```

**Starting code (already in the cell below):**

```python
import importlib
import sys
import tempfile
import tomllib
from pathlib import Path

pyproject = """
[build-system]
requires = ["setuptools>=69"]
build-backend = "setuptools.build_meta"

[project]
name = "tally"
version = "0.3.1"
requires-python = ">=3.12"
dependencies = ["rich>=13", "httpx==0.27.0"]

[project.scripts]
tally = "tally.cli:main"
"""

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
(root / "tally").mkdir()
(root / "tally" / "__init__.py").write_text("")
(root / "tally" / "cli.py").write_text("def main():\n    print('tally is running')\n")
```

<details><summary><b>Hint</b></summary>

`[project.scripts]` becomes `data["project"]["scripts"]`, a dict. `re.split(r"[<>=!~]", text)[0]` keeps the part before the first operator.

</details>

<details><summary><b>Expected output</b></summary>

```text
tally 0.3.1 (python >=3.12)
rich
httpx
tally -> tally.cli:main
tally is running
```

</details>

<details><summary><b>Solution</b></summary>

```python
import re

data = tomllib.loads(pyproject)
project = data["project"]
print(f"{project['name']} {project['version']} (python {project['requires-python']})")
for dep in project["dependencies"]:
    print(re.split(r"[<>=!~]", dep)[0])

sys.path.insert(0, str(root))
for command, target in project["scripts"].items():
    print(command, "->", target)
    module_name, func_name = target.split(":")
    module = importlib.import_module(module_name)
    getattr(module, func_name)()
sys.path.remove(str(root))

workdir.cleanup()
```

`pyproject.toml` is the one file that describes a modern Python project: `[build-system]` says how to build it, `[project]` gives its name, version, supported Python and dependencies, and `[project.scripts]` maps a command name to a `module:function`. When pip installs the project it writes a tiny launcher that does what your code just did: import the module and call the function. Normally you install into a **virtual environment**: `python -m venv .venv` makes a folder with its own Python and its own `site-packages`, so each project's dependencies can't clash with another's. Then `python -m pip install -e .` installs your project in editable mode.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import re
from importlib.metadata import EntryPoint

project = tomllib.loads(pyproject)["project"]
print(f"{project['name']} {project['version']} (python {project['requires-python']})")
for requirement in project["dependencies"]:
    print(re.match(r"[A-Za-z0-9._-]+", requirement).group())

sys.path.insert(0, str(root))
try:
    for name, value in project["scripts"].items():
        print(name, "->", value)
        EntryPoint(name=name, value=value, group="console_scripts").load()()
finally:
    sys.path.remove(str(root))
    workdir.cleanup()
```

`importlib.metadata.EntryPoint.load()` is the standard way to resolve `module:function`, the same machinery plugin systems use. `try`/`finally` restores `sys.path` even if the script crashes. Seniors always run `python -m pip` rather than bare `pip`, so packages go into the interpreter they meant, and pin exact versions in a lock file for deployments.

</details>

In [ ]:
import importlib
import sys
import tempfile
import tomllib
from pathlib import Path

pyproject = """
[build-system]
requires = ["setuptools>=69"]
build-backend = "setuptools.build_meta"

[project]
name = "tally"
version = "0.3.1"
requires-python = ">=3.12"
dependencies = ["rich>=13", "httpx==0.27.0"]

[project.scripts]
tally = "tally.cli:main"
"""

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
(root / "tally").mkdir()
(root / "tally" / "__init__.py").write_text("")
(root / "tally" / "cli.py").write_text("def main():\n    print('tally is running')\n")

# your code here


### NB31-Q24 · L4 · Template strings · Predict [3.14+]

A **t-string** looks like an f-string with a `t` prefix, but it doesn't
produce a `str`. Predict the output.

**What does this print?**

```python
from string.templatelib import Interpolation

user = "<b>Eve</b>"
points = 1250
template = t"Player {user} has {points:,} points"

print(type(template).__name__)
print(template.strings)
print(template.values)
for part in template:
    if isinstance(part, Interpolation):
        print("slot:", part.expression, repr(part.format_spec))
```

<details><summary><b>Hint</b></summary>

A t-string keeps the literal text and the `{...}` values **apart**, and doesn't apply the format spec for you.

</details>

<details><summary><b>Expected output</b></summary>

```text
Template
('Player ', ' has ', ' points')
('<b>Eve</b>', 1250)
slot: user ''
slot: points ','
```

</details>

<details><summary><b>Solution</b></summary>

A t-string evaluates to a `Template` object. `.strings` holds the literal pieces (always one more than the slots), `.values` holds the evaluated expressions, and iterating gives the pieces in order, with each slot as an `Interpolation` that remembers its source text and its format spec (`,`). Nothing has been glued into a string yet. That's the point: code that receives a template decides how each value is inserted, for example by HTML-escaping it or by turning it into a SQL parameter (Q26).

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import html
from string.templatelib import Interpolation


def render_html(template):
    parts = []
    for part in template:
        if isinstance(part, Interpolation):
            parts.append(html.escape(format(part.value, part.format_spec)))
        else:
            parts.append(part)
    return "".join(parts)


user = "<b>Eve</b>"
points = 1250
print(render_html(t"Player {user} has {points:,} points"))
```

A renderer like `render_html` escapes every value but leaves your own literal text alone, so user input can't inject tags into the page. With an f-string the value is already mixed into the text before any function sees it, and escaping the whole string would escape your markup too.

</details>

In [ ]:
# Your prediction (write it here, then open Expected output):
# 

---

## L5 · Hard

### NB31-Q25 · L5 · Layered config · Write

Build the final settings from four layers, each overriding the one before:

1. `DEFAULTS`
2. the `[server]` table in `toml_text`
3. environment variables named `APP_<KEY>` (for example `APP_PORT`)
4. the command-line flags `--host`, `--port`, `--workers` and `--debug`
   (parsed from `argv`)

Values must keep their types: `port` and `workers` are `int`, `debug` is a
`bool`. Print every setting, in the order of `DEFAULTS`, with the layer
it came from:

```text
host = '127.0.0.1' (default)
port = 9100 (env)
debug = False (env)
workers = 8 (cli)
```

Two traps: environment variables are always text (and
`bool("false")` is `True`), and a flag the user didn't type must not
override anything.

**Starting code (already in the cell below):**

```python
import argparse
import os
import tomllib

DEFAULTS = {"host": "127.0.0.1", "port": 8000, "debug": False, "workers": 2}

toml_text = """
[server]
port = 9000
workers = 4
"""

os.environ["APP_PORT"] = "9100"
os.environ["APP_DEBUG"] = "false"
os.environ.pop("APP_HOST", None)
os.environ.pop("APP_WORKERS", None)

argv = ["--workers", "8"]
```

<details><summary><b>Hint</b></summary>

Keep two dicts: `settings` and `source`. Convert env text by looking at the type of the default (check `bool` before `int`: `True` is an `int` too). For the CLI, make every option default to `None` so you can tell "not given" apart.

</details>

<details><summary><b>Expected output</b></summary>

```text
host = '127.0.0.1' (default)
port = 9100 (env)
debug = False (env)
workers = 8 (cli)
```

</details>

<details><summary><b>Solution</b></summary>

```python
settings = dict(DEFAULTS)
source = dict.fromkeys(DEFAULTS, "default")

for key, value in tomllib.loads(toml_text).get("server", {}).items():
    settings[key] = value
    source[key] = "toml"

for key in DEFAULTS:
    raw = os.environ.get("APP_" + key.upper())
    if raw is None:
        continue
    if isinstance(DEFAULTS[key], bool):
        settings[key] = raw.lower() in ("1", "true", "yes", "on")
    elif isinstance(DEFAULTS[key], int):
        settings[key] = int(raw)
    else:
        settings[key] = raw
    source[key] = "env"

parser = argparse.ArgumentParser(prog="server")
parser.add_argument("--host")
parser.add_argument("--port", type=int)
parser.add_argument("--workers", type=int)
parser.add_argument("--debug", action="store_true", default=None)
args = parser.parse_args(argv)
for key, value in vars(args).items():
    if value is not None:
        settings[key] = value
        source[key] = "cli"

for key in DEFAULTS:
    print(f"{key} = {settings[key]!r} ({source[key]})")
```

Each layer only touches the keys it actually provides, so later layers win. TOML values already have types; env values are text, so they're converted using the default's type. `bool` is checked before `int` because `isinstance(False, int)` is `True`. On the command line every option defaults to `None` (and `--debug` uses `default=None`), so an absent flag can't overwrite the env value with argparse's default.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
from collections import ChainMap


def parse_bool(text):
    lowered = text.strip().lower()
    if lowered in ("1", "true", "yes", "on"):
        return True
    if lowered in ("0", "false", "no", "off"):
        return False
    raise ValueError(f"not a boolean: {text!r}")


CONVERT = {"host": str, "port": int, "workers": int, "debug": parse_bool}

file_layer = tomllib.loads(toml_text).get("server", {})
env_layer = {key: CONVERT[key](os.environ[f"APP_{key.upper()}"])
             for key in DEFAULTS if f"APP_{key.upper()}" in os.environ}

parser = argparse.ArgumentParser(prog="server")
parser.add_argument("--host")
parser.add_argument("--port", type=int)
parser.add_argument("--workers", type=int)
parser.add_argument("--debug", action=argparse.BooleanOptionalAction)
cli_layer = {k: v for k, v in vars(parser.parse_args(argv)).items() if v is not None}

layers = {"cli": cli_layer, "env": env_layer, "toml": file_layer, "default": DEFAULTS}
settings = ChainMap(*layers.values())
for key in DEFAULTS:
    origin = next(name for name, layer in layers.items() if key in layer)
    print(f"{key} = {settings[key]!r} ({origin})")
```

`ChainMap` (NB15) looks a key up in each layer in turn, which is exactly "later layers win" with no copying. `parse_bool` **rejects** a typo like `APP_DEBUG=flase` instead of silently treating it as `False`. `BooleanOptionalAction` adds both `--debug` and `--no-debug`, so the command line can switch off a debug mode the environment switched on.

</details>

In [ ]:
import argparse
import os
import tomllib

DEFAULTS = {"host": "127.0.0.1", "port": 8000, "debug": False, "workers": 2}

toml_text = """
[server]
port = 9000
workers = 4
"""

os.environ["APP_PORT"] = "9100"
os.environ["APP_DEBUG"] = "false"
os.environ.pop("APP_HOST", None)
os.environ.pop("APP_WORKERS", None)

argv = ["--workers", "8"]

# your code here


### NB31-Q26 · L5 · Template strings · Write [3.14+]

f-strings make SQL injection easy (Q19). Template strings let you keep the
f-string look while staying safe. Write:

- `sql(template)`: turns a t-string into a `(query, params)` pair. Literal
  text is kept, each `{...}` becomes `?`, and its value is appended to the
  `params` list. If it gets anything that isn't a `Template` (such as an
  f-string), it raises `TypeError`.
- `query(template)`: runs it on `conn` and returns all rows.

Then print `sql(...)` and `query(...)` for the books by `author` after
the year `after`, print `query(...)` for books whose author is `attack`,
and finally call `query` with an **f-string** version of that attack,
printing `refused: not a template` when it raises `TypeError`.

```text
('SELECT title FROM books WHERE author = ? AND year > ?', ['Frank Herbert', 1970])
[('Children of Dune',)]
[]
refused: not a template
```

**Starting code (already in the cell below):**

```python
import sqlite3
from string.templatelib import Interpolation, Template

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE books (title TEXT, author TEXT, year INTEGER)")
conn.executemany("INSERT INTO books VALUES (?, ?, ?)", [
    ("Dune", "Frank Herbert", 1965),
    ("Hyperion", "Dan Simmons", 1989),
    ("Children of Dune", "Frank Herbert", 1976),
])

author = "Frank Herbert"
after = 1970
attack = "x' OR '1'='1"
```

<details><summary><b>Hint</b></summary>

Loop over the template: a `str` part is added to the query text as-is, an `Interpolation` adds `"?"` to the text and `part.value` to the params.

</details>

<details><summary><b>Expected output</b></summary>

```text
('SELECT title FROM books WHERE author = ? AND year > ?', ['Frank Herbert', 1970])
[('Children of Dune',)]
[]
refused: not a template
```

</details>

<details><summary><b>Solution</b></summary>

```python
def sql(template):
    if not isinstance(template, Template):
        raise TypeError("sql() needs a t-string")
    text = ""
    params = []
    for part in template:
        if isinstance(part, Interpolation):
            text += "?"
            params.append(part.value)
        else:
            text += part
    return text, params


def query(template):
    text, params = sql(template)
    return conn.execute(text, params).fetchall()


print(sql(t"SELECT title FROM books WHERE author = {author} AND year > {after}"))
print(query(t"SELECT title FROM books WHERE author = {author} AND year > {after}"))
print(query(t"SELECT title FROM books WHERE author = {attack}"))
try:
    query(f"SELECT title FROM books WHERE author = '{attack}'")
except TypeError:
    print("refused: not a template")
```

The caller writes something that reads like an f-string, but the values never touch the SQL text: each one becomes a `?` and travels as a parameter, so the attack string is compared as plain text and matches nothing. The `isinstance` check is what makes the design safe: an f-string has **already** mixed the value into the text, and nothing can separate them again, so it has to be refused outright.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
def sql(template: Template) -> tuple[str, list[object]]:
    """Turn a t-string into a parameterised query. Values never enter the SQL text."""
    if not isinstance(template, Template):
        raise TypeError(f"sql() needs a t-string, got {type(template).__name__}")
    for slot in template.interpolations:
        if slot.conversion or slot.format_spec:
            raise ValueError(f"no !r or :spec allowed in SQL slot {{{slot.expression}}}")
    return "?".join(template.strings), list(template.values)


def query(template: Template) -> list[tuple]:
    return conn.execute(*sql(template)).fetchall()


print(sql(t"SELECT title FROM books WHERE author = {author} AND year > {after}"))
print(query(t"SELECT title FROM books WHERE author = {author} AND year > {after}"))
print(query(t"SELECT title FROM books WHERE author = {attack}"))
try:
    query(f"SELECT title FROM books WHERE author = '{attack}'")
except TypeError:
    print("refused: not a template")
```

`template.strings` always has exactly one more item than `template.values`, so `"?".join(strings)` rebuilds the query in one step. The senior also rejects `{x!r}` and `{x:...}` in SQL slots, because formatting a value before it becomes a parameter would defeat the purpose. Type hints (NB26) make the "t-strings only" rule visible to editors.

</details>

In [ ]:
import sqlite3
from string.templatelib import Interpolation, Template

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE books (title TEXT, author TEXT, year INTEGER)")
conn.executemany("INSERT INTO books VALUES (?, ?, ?)", [
    ("Dune", "Frank Herbert", 1965),
    ("Hyperion", "Dan Simmons", 1989),
    ("Children of Dune", "Frank Herbert", 1976),
])

author = "Frank Herbert"
after = 1970
attack = "x' OR '1'='1"

# your code here


### NB31-Q27 · L5 · subprocess · Fix

This report should count the lines in two files, but `jan sales.csv`
silently disappears from the output. Fix **both** problems: the way the
command is built, and the fact that a failing child goes unnoticed.

**Buggy code (copied into the cell below):**

```python
import subprocess
import sys
import tempfile
from pathlib import Path

COUNTER = """
import sys
from pathlib import Path

for arg in sys.argv[1:]:
    path = Path(arg)
    lines = path.read_text().splitlines()
    print(f"{path.name}: {len(lines)} lines")
"""

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    script = root / "count_lines.py"
    script.write_text(COUNTER)
    (root / "jan sales.csv").write_text("a\nb\nc\n")
    (root / "feb.csv").write_text("a\nb\n")

    for name in ["jan sales.csv", "feb.csv"]:
        command = f"{sys.executable} {script} {root / name}"
        result = subprocess.run(command.split(), capture_output=True, text=True)
        print(result.stdout, end="")
```

<details><summary><b>Hint</b></summary>

Print `command.split()` for the first file: how many arguments does the child get? Then: what does `subprocess.run` do by default when the child fails?

</details>

<details><summary><b>Expected output</b></summary>

```text
jan sales.csv: 3 lines
feb.csv: 2 lines
```

</details>

<details><summary><b>Solution</b></summary>

```python
import subprocess
import sys
import tempfile
from pathlib import Path

COUNTER = """
import sys
from pathlib import Path

for arg in sys.argv[1:]:
    path = Path(arg)
    lines = path.read_text().splitlines()
    print(f"{path.name}: {len(lines)} lines")
"""

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    script = root / "count_lines.py"
    script.write_text(COUNTER)
    (root / "jan sales.csv").write_text("a\nb\nc\n")
    (root / "feb.csv").write_text("a\nb\n")

    for name in ["jan sales.csv", "feb.csv"]:
        result = subprocess.run([sys.executable, script, root / name],
                                capture_output=True, text=True, check=True)
        print(result.stdout, end="")
```

Bug 1: splitting the command string on spaces cut `jan sales.csv` into two arguments, `.../jan` and `sales.csv`, so the child crashed with `FileNotFoundError`. A list passes each argument whole, spaces and all (`Path` objects are accepted too). Bug 2: the child's failure was invisible because nobody checked its exit code; its traceback went to the captured stderr and was thrown away. `check=True` would have turned it into a loud `CalledProcessError`.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import subprocess
import sys
import tempfile
from pathlib import Path

COUNTER = """
import sys
from pathlib import Path

for arg in sys.argv[1:]:
    path = Path(arg)
    lines = path.read_text().splitlines()
    print(f"{path.name}: {len(lines)} lines")
"""

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    script = root / "count_lines.py"
    script.write_text(COUNTER)
    (root / "jan sales.csv").write_text("a\nb\nc\n")
    (root / "feb.csv").write_text("a\nb\n")

    files = [root / "jan sales.csv", root / "feb.csv"]
    result = subprocess.run([sys.executable, script, *files],
                            capture_output=True, text=True, check=True, timeout=30)
    print(result.stdout, end="")
```

Starting a process is slow (tens of milliseconds), so the senior passes every file to **one** child, which the script already supports. `*files` unpacks the list into separate arguments. A timeout guards against a hung child.

</details>

In [ ]:
import subprocess
import sys
import tempfile
from pathlib import Path

COUNTER = """
import sys
from pathlib import Path

for arg in sys.argv[1:]:
    path = Path(arg)
    lines = path.read_text().splitlines()
    print(f"{path.name}: {len(lines)} lines")
"""

with tempfile.TemporaryDirectory() as tmp:
    root = Path(tmp)
    script = root / "count_lines.py"
    script.write_text(COUNTER)
    (root / "jan sales.csv").write_text("a\nb\nc\n")
    (root / "feb.csv").write_text("a\nb\n")

    for name in ["jan sales.csv", "feb.csv"]:
        command = f"{sys.executable} {script} {root / name}"
        result = subprocess.run(command.split(), capture_output=True, text=True)
        print(result.stdout, end="")


### NB31-Q28 · L5 · shutil and zipfile · Write

Back up `project` and prove the backup is good:

1. Copy `project` to `root / "staging"` with `shutil.copytree`, skipping
   every `__pycache__` folder (`shutil.ignore_patterns`).
2. Zip every **file** under staging into `root / "backup.zip"`, using
   paths relative to staging with `/` separators as member names.
3. Print the archive's sorted member names, then extract it into
   `root / "restored"`.
4. For every file under `project` outside `__pycache__` (sorted), compare
   the SHA-256 hash of the original with the restored copy and print
   `path OK` or `path MISMATCH`, then a summary line. Clean up.

```text
['README.md', 'src/app.py']
README.md OK
src/app.py OK
2 files verified, 0 mismatches
```

**Starting code (already in the cell below):**

```python
import hashlib
import shutil
import tempfile
import zipfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
project = root / "project"
(project / "src" / "__pycache__").mkdir(parents=True)
(project / "src" / "app.py").write_text("print('hi')\n")
(project / "src" / "__pycache__" / "app.cpython-314.pyc").write_bytes(b"\x00\x01")
(project / "README.md").write_text("# Project\n")
```

<details><summary><b>Hint</b></summary>

`path.relative_to(staging).as_posix()` gives a member name. `hashlib.sha256(data).hexdigest()` hashes bytes. `"__pycache__" in path.parts` tells you whether a path is inside one.

</details>

<details><summary><b>Expected output</b></summary>

```text
['README.md', 'src/app.py']
README.md OK
src/app.py OK
2 files verified, 0 mismatches
```

</details>

<details><summary><b>Solution</b></summary>

```python
staging = root / "staging"
shutil.copytree(project, staging, ignore=shutil.ignore_patterns("__pycache__"))

archive = root / "backup.zip"
with zipfile.ZipFile(archive, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for path in sorted(staging.rglob("*")):
        if path.is_file():
            zf.write(path, arcname=path.relative_to(staging).as_posix())

restored = root / "restored"
with zipfile.ZipFile(archive) as zf:
    print(sorted(zf.namelist()))
    zf.extractall(restored)


def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()


checked = 0
mismatches = 0
for path in sorted(project.rglob("*")):
    if path.is_dir() or "__pycache__" in path.parts:
        continue
    relative = path.relative_to(project)
    same = sha256(path) == sha256(restored / relative)
    print(relative.as_posix(), "OK" if same else "MISMATCH")
    checked += 1
    if not same:
        mismatches += 1
print(f"{checked} files verified, {mismatches} mismatches")

workdir.cleanup()
```

A backup you've never restored is only a hope. Comparing **hashes** proves each restored file is byte-for-byte identical without comparing the files themselves side by side. The traps: forgetting to skip `__pycache__` (bytecode doesn't belong in a backup), storing absolute paths in the zip, and using `\` separators on Windows. Zip member names always use `/`, which `as_posix()` guarantees.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
staging = root / "staging"
shutil.copytree(project, staging, ignore=shutil.ignore_patterns("__pycache__"))
archive = shutil.make_archive(str(root / "backup"), "zip", root_dir=staging)

with zipfile.ZipFile(archive) as zf:
    print(sorted(name for name in zf.namelist() if not name.endswith("/")))
restored = root / "restored"
shutil.unpack_archive(archive, restored)


def digest(path):
    with open(path, "rb") as f:
        return hashlib.file_digest(f, "sha256").hexdigest()


originals = sorted(p for p in project.rglob("*") if p.is_file() and "__pycache__" not in p.parts)
mismatches = 0
for path in originals:
    relative = path.relative_to(project)
    same = digest(path) == digest(restored / relative)
    mismatches += not same
    print(relative.as_posix(), "OK" if same else "MISMATCH")
print(f"{len(originals)} files verified, {mismatches} mismatches")

workdir.cleanup()
```

`shutil.make_archive` and `unpack_archive` do the zip work in one line each (they also add folder entries like `src/`, filtered out here). `hashlib.file_digest` (3.11+) reads the file in chunks, so hashing a 10 GB file doesn't need 10 GB of memory, unlike `read_bytes()`.

</details>

In [ ]:
import hashlib
import shutil
import tempfile
import zipfile
from pathlib import Path

workdir = tempfile.TemporaryDirectory()
root = Path(workdir.name)
project = root / "project"
(project / "src" / "__pycache__").mkdir(parents=True)
(project / "src" / "app.py").write_text("print('hi')\n")
(project / "src" / "__pycache__" / "app.cpython-314.pyc").write_bytes(b"\x00\x01")
(project / "README.md").write_text("# Project\n")

# your code here


---

## Mini-project

Pull together everything from this notebook.

## NB31-P · L5 · Mini-project: Expense Tracker CLI · Write

Build an expense tracker that runs like a command-line tool. Simulate a
terminal session by calling `main(argv)` for each list in `RUNS`, printing
`$ expenses ` plus the arguments before each run, and collecting every
exit code.

Requirements:

- **Logging:** `logging.basicConfig(stream=sys.stdout, force=True,
  level=logging.INFO, format="%(levelname)s %(message)s")` and a logger
  named `"expenses"`.
- **Database:** one `sqlite3` connection to `":memory:"` with a table
  `expenses(id INTEGER PRIMARY KEY, cents INTEGER, category TEXT, note TEXT)`.
  Store money as whole cents (`round(amount * 100)`). Every value goes in
  through `?` placeholders, and every write runs inside `with conn:`.
- **argparse** (`prog="expenses"`), subcommands:
    - `add amount category [--note NOTE]`: amount must be positive, else log
      an error and return `1`. On success log `added #id amount category`.
    - `list [--category C]`: print `#id  amount  category  note` rows by id.
    - `summary`: total per category (alphabetical), then `TOTAL`.
    - `delete id`: log `deleted #id`, or a warning if there's no such id.
- **Bad commands:** argparse raises `SystemExit`. Catch it, hide argparse's
  own stderr message with `contextlib.redirect_stderr(io.StringIO())`,
  log `bad command: <the arguments>` and return `2`.
- `main` returns `0` on success. After all runs, print the exit codes.

Money is shown with two decimals; `list` and `summary` use the widths
below.

```text
$ expenses add 12.50 food --note lunch
INFO added #1 12.50 food
$ expenses add 40 transport
INFO added #2 40.00 transport
$ expenses add -5 food
ERROR amount must be positive, got -5.00
$ expenses add 8.25 food --note coffee'); DROP TABLE expenses; --
INFO added #3 8.25 food
$ expenses list --category food
#1    12.50  food       lunch
#3     8.25  food       coffee'); DROP TABLE expenses; --
$ expenses summary
food          20.75
transport     40.00
TOTAL         60.75
$ expenses delete 1
INFO deleted #1
$ expenses delete 99
WARNING no expense #99
$ expenses refund 3
ERROR bad command: refund 3
$ expenses summary
food           8.25
transport     40.00
TOTAL         48.25
exit codes: [0, 0, 1, 0, 0, 0, 0, 0, 2, 0]
```

Row formats: `f"#{id:<4}{amount:>6.2f}  {category:<9}  {note}"` for `list`,
and `f"{name:<10}{amount:>9.2f}"` for `summary`.

**Starting code (already in the cell below):**

```python
import argparse
import contextlib
import io
import logging
import sqlite3
import sys

RUNS = [
    ["add", "12.50", "food", "--note", "lunch"],
    ["add", "40", "transport"],
    ["add", "-5", "food"],
    ["add", "8.25", "food", "--note", "coffee'); DROP TABLE expenses; --"],
    ["list", "--category", "food"],
    ["summary"],
    ["delete", "1"],
    ["delete", "99"],
    ["refund", "3"],
    ["summary"],
]
```

<details><summary><b>Hint</b></summary>

Build the parser once, outside `main`. Inside `main`: parse (catching `SystemExit`), then branch on `args.command`. For `list`, build the `WHERE` part from fixed text and pass the category as a parameter. `GROUP BY category` with `SUM(cents)` does the summary.

</details>

<details><summary><b>Expected output</b></summary>

```text
$ expenses add 12.50 food --note lunch
INFO added #1 12.50 food
$ expenses add 40 transport
INFO added #2 40.00 transport
$ expenses add -5 food
ERROR amount must be positive, got -5.00
$ expenses add 8.25 food --note coffee'); DROP TABLE expenses; --
INFO added #3 8.25 food
$ expenses list --category food
#1    12.50  food       lunch
#3     8.25  food       coffee'); DROP TABLE expenses; --
$ expenses summary
food          20.75
transport     40.00
TOTAL         60.75
$ expenses delete 1
INFO deleted #1
$ expenses delete 99
WARNING no expense #99
$ expenses refund 3
ERROR bad command: refund 3
$ expenses summary
food           8.25
transport     40.00
TOTAL         48.25
exit codes: [0, 0, 1, 0, 0, 0, 0, 0, 2, 0]
```

</details>

<details><summary><b>Solution</b></summary>

```python
logging.basicConfig(stream=sys.stdout, force=True, level=logging.INFO,
                    format="%(levelname)s %(message)s")
log = logging.getLogger("expenses")

conn = sqlite3.connect(":memory:")
conn.execute("CREATE TABLE expenses (id INTEGER PRIMARY KEY, cents INTEGER, "
             "category TEXT, note TEXT)")

parser = argparse.ArgumentParser(prog="expenses")
sub = parser.add_subparsers(dest="command", required=True)
add = sub.add_parser("add")
add.add_argument("amount", type=float)
add.add_argument("category")
add.add_argument("--note", default="")
lst = sub.add_parser("list")
lst.add_argument("--category")
sub.add_parser("summary")
delete = sub.add_parser("delete")
delete.add_argument("id", type=int)


def main(argv):
    try:
        with contextlib.redirect_stderr(io.StringIO()):
            args = parser.parse_args(argv)
    except SystemExit:
        log.error("bad command: %s", " ".join(argv))
        return 2

    if args.command == "add":
        if args.amount <= 0:
            log.error("amount must be positive, got %.2f", args.amount)
            return 1
        with conn:
            cursor = conn.execute(
                "INSERT INTO expenses (cents, category, note) VALUES (?, ?, ?)",
                (round(args.amount * 100), args.category, args.note))
        log.info("added #%d %.2f %s", cursor.lastrowid, args.amount, args.category)
    elif args.command == "list":
        if args.category is None:
            rows = conn.execute("SELECT id, cents, category, note FROM expenses ORDER BY id")
        else:
            rows = conn.execute("SELECT id, cents, category, note FROM expenses "
                                "WHERE category = ? ORDER BY id", (args.category,))
        for expense_id, cents, category, note in rows:
            print(f"#{expense_id:<4}{cents / 100:>6.2f}  {category:<9}  {note}")
    elif args.command == "summary":
        total = 0
        for category, cents in conn.execute(
                "SELECT category, SUM(cents) FROM expenses GROUP BY category ORDER BY category"):
            print(f"{category:<10}{cents / 100:>9.2f}")
            total += cents
        print(f"{'TOTAL':<10}{total / 100:>9.2f}")
    elif args.command == "delete":
        with conn:
            cursor = conn.execute("DELETE FROM expenses WHERE id = ?", (args.id,))
        if cursor.rowcount == 0:
            log.warning("no expense #%d", args.id)
        else:
            log.info("deleted #%d", args.id)
    return 0


codes = []
for argv in RUNS:
    print("$ expenses", " ".join(argv))
    codes.append(main(argv))
print("exit codes:", codes)
```

The program has the shape of every real CLI: configure logging once, open resources once, build the parser once, and keep `main(argv)` free of global input so it can be driven by a list (here) or by `sys.argv` (in a terminal). Money is stored as integer cents, so `12.50 + 8.25` can't drift. The note full of SQL is stored and printed as plain text because it only ever travels as a parameter. Returning exit codes instead of printing "error" lets shell scripts and CI react to failures. `-5` is still read as a positional value because the parser has no options that look like negative numbers.

</details>

<details><summary><b>Senior dev solution</b></summary>

```python
import contextlib
import io
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation

logging.basicConfig(stream=sys.stdout, force=True, level=logging.INFO,
                    format="%(levelname)s %(message)s")
log = logging.getLogger("expenses")

CENT = Decimal("0.01")


def money(text: str) -> Decimal:
    try:
        return Decimal(text).quantize(CENT)
    except InvalidOperation:
        raise argparse.ArgumentTypeError(f"not an amount: {text!r}") from None


@dataclass(frozen=True)
class Expense:
    id: int
    cents: int
    category: str
    note: str

    def line(self) -> str:
        return f"#{self.id:<4}{self.cents / 100:>6.2f}  {self.category:<9}  {self.note}"


class ExpenseStore:
    def __init__(self, conn: sqlite3.Connection) -> None:
        self.conn = conn
        conn.execute("""
            CREATE TABLE IF NOT EXISTS expenses (
                id       INTEGER PRIMARY KEY,
                cents    INTEGER NOT NULL CHECK (cents > 0),
                category TEXT    NOT NULL,
                note     TEXT    NOT NULL DEFAULT ''
            )""")

    def add(self, cents: int, category: str, note: str) -> int:
        with self.conn:
            return self.conn.execute(
                "INSERT INTO expenses (cents, category, note) VALUES (?, ?, ?)",
                (cents, category, note)).lastrowid

    def find(self, category: str | None = None) -> list[Expense]:
        sql = "SELECT id, cents, category, note FROM expenses"
        params: tuple = ()
        if category is not None:
            sql += " WHERE category = ?"
            params = (category,)
        return [Expense(*row) for row in self.conn.execute(sql + " ORDER BY id", params)]

    def totals(self) -> list[tuple[str, int]]:
        return self.conn.execute("SELECT category, SUM(cents) FROM expenses "
                                 "GROUP BY category ORDER BY category").fetchall()

    def delete(self, expense_id: int) -> bool:
        with self.conn:
            return self.conn.execute("DELETE FROM expenses WHERE id = ?",
                                     (expense_id,)).rowcount == 1


def cmd_add(store: ExpenseStore, args) -> int:
    if args.amount <= 0:
        log.error("amount must be positive, got %s", args.amount)
        return 1
    new_id = store.add(int(args.amount * 100), args.category, args.note)
    log.info("added #%d %s %s", new_id, args.amount, args.category)
    return 0


def cmd_list(store: ExpenseStore, args) -> int:
    for expense in store.find(args.category):
        print(expense.line())
    return 0


def cmd_summary(store: ExpenseStore, args) -> int:
    totals = store.totals()
    for category, cents in totals:
        print(f"{category:<10}{cents / 100:>9.2f}")
    print(f"{'TOTAL':<10}{sum(c for _, c in totals) / 100:>9.2f}")
    return 0


def cmd_delete(store: ExpenseStore, args) -> int:
    if store.delete(args.id):
        log.info("deleted #%d", args.id)
    else:
        log.warning("no expense #%d", args.id)
    return 0


def build_parser() -> argparse.ArgumentParser:
    parser = argparse.ArgumentParser(prog="expenses", description="Track spending.")
    sub = parser.add_subparsers(dest="command", required=True)
    add = sub.add_parser("add", help="record an expense")
    add.add_argument("amount", type=money)
    add.add_argument("category")
    add.add_argument("--note", default="")
    add.set_defaults(handler=cmd_add)
    lst = sub.add_parser("list", help="show expenses")
    lst.add_argument("--category")
    lst.set_defaults(handler=cmd_list)
    sub.add_parser("summary", help="totals per category").set_defaults(handler=cmd_summary)
    delete = sub.add_parser("delete", help="remove an expense")
    delete.add_argument("id", type=int)
    delete.set_defaults(handler=cmd_delete)
    return parser


def main(argv: list[str], store: ExpenseStore, parser: argparse.ArgumentParser) -> int:
    try:
        with contextlib.redirect_stderr(io.StringIO()):
            args = parser.parse_args(argv)
    except SystemExit as exc:
        log.error("bad command: %s", " ".join(argv))
        return exc.code if isinstance(exc.code, int) else 2
    return args.handler(store, args)


store = ExpenseStore(sqlite3.connect(":memory:"))
parser = build_parser()
codes = []
for argv in RUNS:
    print("$ expenses", " ".join(argv))
    codes.append(main(argv, store, parser))
print("exit codes:", codes)
```

The senior separates layers: `ExpenseStore` is the only code that knows SQL, the `cmd_*` functions hold the behaviour, and argparse wiring lives in `build_parser`. Each piece can be tested alone (NB28) with an in-memory store. Amounts are parsed straight from text into `Decimal` (NB16), so `int(amount * 100)` is exact, and the database's `CHECK (cents > 0)` backs up the Python check. `set_defaults(handler=...)` replaces the `if`/`elif` chain, and dependencies (`store`, `parser`) are passed in rather than read from globals.

</details>

In [ ]:
import argparse
import contextlib
import io
import logging
import sqlite3
import sys

RUNS = [
    ["add", "12.50", "food", "--note", "lunch"],
    ["add", "40", "transport"],
    ["add", "-5", "food"],
    ["add", "8.25", "food", "--note", "coffee'); DROP TABLE expenses; --"],
    ["list", "--category", "food"],
    ["summary"],
    ["delete", "1"],
    ["delete", "99"],
    ["refund", "3"],
    ["summary"],
]

# your code here


---

## Done

Next up: **NB32 · Advanced Capstone** in `advanced/`.